# PLEASE DO NOT DELETE


My name's Tran Minh Duong, i'm coding for my personal project, writing reports to Dr. Quang, please don't delete this file :((


In [1]:
!source ./miniconda3/bin/activate
!conda --version

conda 24.11.3


In [2]:
!nvidia-smi

Mon Jun  9 10:22:08 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.01             Driver Version: 535.183.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090        On  | 00000000:18:00.0 Off |                  N/A |
| 41%   29C    P8              23W / 350W |      1MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
# PyTorch 2.5.1 with CUDA 12.1
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Hugging Face Transformers and Accelerate
!pip install transformers accelerate num2words


Looking in indexes: https://download.pytorch.org/whl/cu121


In [4]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.5.1+cu121
12.1
True


In [4]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("USING DEVICE: ",device)

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct",use_fast=True)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa"
)
print("Success")

/home/student4/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


USING DEVICE:  cuda


Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.76s/it]


Success


In [5]:
import time
messages = [
    {
        "role":"user",
        "content":[
            {
                "type":"image",
                "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
            },
            {
                "type":"text",
                "text":"Please describe this image"
            }
        ]
    }

]

start = time.time()

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(device)

generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
       generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)
print("Inference time:",time.time() - start,"seconds")

["The image depicts a small, fluffy animal walking on a snowy surface. The animal has a thick, dense coat of fur that appears to be well-suited for cold weather. Its fur is primarily brown with some darker patches, and it has a bushy tail. The animal's ears are small and rounded, and its eyes are partially obscured by the fur. The background features a chain-link fence and several tall, white birch trees, indicating that the setting is likely an enclosure or a zoo. The snow on the ground suggests that it is winter."]
Inference time: 7.523715019226074 seconds


In [6]:
from PIL import Image
import requests
import torch
from torchvision import io
from typing import Dict
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor

# Load the model in half-precision on the available device(s)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct", torch_dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

# Image
url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
image = Image.open(requests.get(url, stream=True).raw)

conversation = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]


# Preprocess the inputs
text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
# Excepted output: '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>Describe this image.<|im_end|>\n<|im_start|>assistant\n'

inputs = processor(
    text=[text_prompt], images=[image], padding=True, return_tensors="pt"
)
inputs = inputs.to("cuda")

# Inference: Generation of the output
output_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids = [
    output_ids[len(input_ids) :]
    for input_ids, output_ids in zip(inputs.input_ids, output_ids)
]
output_text = processor.batch_decode(
    generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
)
print(output_text)


Fetching 2 files:   0%|          | 0/2 [01:48<?, ?it/s]


KeyboardInterrupt: 

# SMolVLM2

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-256M-Video-Instruct")
model = AutoModelForImageTextToText.from_pretrained(
    "HuggingFaceTB/SmolVLM2-256M-Video-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)
print("SmolVLM2 loaded succesfully")

/home/student4/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SmolVLM2 loaded succesfully


In [7]:
import time

template = """
You are a visual analyst evaluating an image for signs of fire and the surrounding context. Do the following tasks:
1: Summarize what you see in the image. Describe the environment, key objects, people, and any signs of fire or smoke.
2: Based on your summary, decide what types of fire situation: no fire, controlled fire (e.g., fireplace, campfire, cooking) or a dangerous/uncontrolled fire (e.g., house fire, forest fire)?
"""



conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image",
             "url": "https://assets.site-static.com/userFiles/2566/image/how-to-design-the-room-around-your-fireplace.jpg"},
            {"type": "text",
             "text": template}
        ]
    }
]

start_time = time.time()

inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

output_ids = model.generate(**inputs, max_new_tokens=512)
prompt_len = inputs["input_ids"].shape[1]
gen_tokens = output_ids[0, prompt_len:]
final_answer = processor.tokenizer.decode(
    gen_tokens,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
).strip()

print(final_answer)
print(f"Inference time: {(time.time() - start_time):.4f}s")

### Summary:
The image depicts a modern living room with large windows allowing natural light to flood the space. The room features two gray sofas adorned with various pillows, including patterned and solid colors. In the center of the room is a glass coffee table with a few items on it, including a teapot and some books. Above the coffee table is a white fireplace with a brick surround, which currently has a fire burning inside. The fireplace is flanked by shelves that hold decorative items such as vases, small sculptures, and plants. The walls are painted white, and there are additional shelves above the fireplace holding more decorative pieces. The overall ambiance is cozy and well-decorated.

### Analysis:
Based on the summary, the fire situation appears to be a controlled fire. The fire is contained within the fireplace, and there are no visible signs of smoke or flames outside the fireplace area. The fire is likely intended for warmth and ambiance, rather than being a dangerous o